# SNA Media Sosial untuk MK Metode Penelitian Digital
Prototipe: analisis jaringan aktor (Social Network Analysis) dari media sosial & portal berita, dengan bantuan AI untuk draf hasil-pembahasan.

## Cakupan & keterbatasan (baca dulu sebelum pakai)

| Sumber | Status | Catatan |
|---|---|---|
| YouTube | Gratis | Kuota API resmi Google, cukup untuk tugas kelas |
| Portal berita | Gratis (scraping) | Template umum, selector HTML perlu disesuaikan per situs |
| X / Twitter | **Berbayar** | Lewat reseller pihak ketiga (TwitterAPI.io), bukan API resmi X. Biaya per-request |
| Facebook / Instagram | **Tidak disertakan** | Tidak ada jalur gratis/murah setara. CrowdTangle sudah tutup (Agustus 2024). Jalur resmi (Meta Content Library) butuh pengajuan institusi & persetujuan, bukan instan. Scraping langsung berisiko ToS dan sangat rapuh — sengaja tidak dibuatkan kodenya. |

### Tentang modul AI di notebook ini
Modul AI di bagian akhir **membuat draf** bagian Hasil & Pembahasan dari data yang sudah dianalisis di notebook ini sendiri — bukan artikel jurnal siap kirim. AI generatif punya kecenderungan mengarang kutipan/rujukan yang kedengarannya meyakinkan tapi fiktif (hallucination). Karena itu:
- Draf ditandai jelas sebagai draf yang wajib diverifikasi manusia.
- Prompt melarang AI mengarang sitasi — bagian rujukan dikosongkan dengan placeholder untuk diisi manual oleh peneliti.
- Cek kebijakan jurnal tujuan soal pengungkapan (disclosure) penggunaan AI dalam penulisan.

### API key
Notebook ini meminta API key milik masing-masing pengguna (bukan dibagikan bersama) supaya kuota/biaya tidak tercampur antar mahasiswa. Key hanya disimpan di memori sesi Colab-mu sendiri, tidak pernah ditulis ke file.


In [ ]:
!pip install -q requests pandas networkx python-louvain pyvis google-api-python-client beautifulsoup4 lxml anthropic

## 1. Masukkan API key kamu sendiri
Jalankan sel di bawah. Input tidak akan tertampil di layar (pakai getpass). Kosongkan + Enter untuk bagian yang tidak kamu pakai.

In [ ]:
from getpass import getpass

TWITTERAPI_IO_KEY = getpass("TwitterAPI.io API key (kosongkan jika tidak pakai Twitter): ")
YOUTUBE_API_KEY = getpass("YouTube Data API key (kosongkan jika tidak pakai YouTube): ")
ANTHROPIC_API_KEY = getpass("Anthropic (Claude) API key untuk modul AI (kosongkan jika tidak pakai): ")

## 2. Konfigurasi topik riset

In [ ]:
TOPIC_QUERY = "isi kata kunci topik riset di sini"   # contoh: "MBG lang:id"
MAX_PAGES_TWITTER = 5     # tiap page berisi ~20 tweet -> perkirakan biaya sebelum menaikkan ini
YOUTUBE_VIDEO_IDS = []    # isi ID video YouTube yang mau diambil komentarnya, contoh: ["dQw4w9WgXcQ"]
NEWS_URLS = []            # isi daftar URL artikel berita yang mau diambil komentarnya

## 3. Ambil data X/Twitter (via TwitterAPI.io — BERBAYAR)
Setiap pemanggilan `advanced_search` mengembalikan sekitar 20 tweet per halaman dan dikenakan biaya sesuai tarif berjalan di dashboard TwitterAPI.io. `MAX_PAGES_TWITTER` di atas membatasi jumlah halaman supaya biaya terkontrol. Layanan ini beroperasi dengan mengambil data X secara komersial di luar jalur resmi X — cek dashboard-mu untuk tarif dan status layanan terkini. Lewati sel ini kalau tidak pakai Twitter.

In [ ]:
import requests
import time
import pandas as pd

TWITTERAPI_BASE = "https://api.twitterapi.io"

def search_tweets_twitterapi(query, query_type="Latest", max_pages=5, api_key=TWITTERAPI_IO_KEY):
    if not api_key:
        print("Lewati: TwitterAPI.io key kosong.")
        return pd.DataFrame()
    headers = {"x-api-key": api_key}
    all_tweets = []
    cursor = ""
    for _ in range(max_pages):
        params = {"query": query, "queryType": query_type, "cursor": cursor}
        resp = requests.get(f"{TWITTERAPI_BASE}/twitter/tweet/advanced_search", headers=headers, params=params)
        resp.raise_for_status()
        data = resp.json()
        for t in data.get("tweets", []):
            all_tweets.append({
                "id": t.get("id"),
                "text": t.get("text"),
                "author": (t.get("author") or {}).get("userName"),
                "in_reply_to_username": t.get("inReplyToUsername"),
                "conversation_id": t.get("conversationId"),
                "is_reply": t.get("isReply"),
                "created_at": t.get("createdAt"),
                "retweet_count": t.get("retweetCount"),
                "like_count": t.get("likeCount"),
                "reply_count": t.get("replyCount"),
                "mentions": [m.get("screen_name") for m in (t.get("entities") or {}).get("user_mentions", [])],
            })
        if not data.get("has_next_page"):
            break
        cursor = data.get("next_cursor", "")
        time.sleep(1)  # jaga rate limit
    return pd.DataFrame(all_tweets)

df_twitter = search_tweets_twitterapi(TOPIC_QUERY, max_pages=MAX_PAGES_TWITTER)
df_twitter.head()

## 4. Ambil komentar YouTube (gratis, kuota harian Google)

In [ ]:
from googleapiclient.discovery import build

def get_youtube_comments(video_id, api_key=YOUTUBE_API_KEY, max_results=200):
    if not api_key:
        print("Lewati: YouTube API key kosong.")
        return pd.DataFrame()
    youtube = build("youtube", "v3", developerKey=api_key)
    comments = []
    next_page_token = None
    while len(comments) < max_results:
        req = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            pageToken=next_page_token,
            textFormat="plainText",
        )
        res = req.execute()
        for item in res.get("items", []):
            top = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "comment_id": item["snippet"]["topLevelComment"]["id"],
                "author": top.get("authorDisplayName"),
                "text": top.get("textDisplay"),
                "like_count": top.get("likeCount"),
                "published_at": top.get("publishedAt"),
                "parent_id": None,
            })
            for reply in item.get("replies", {}).get("comments", []):
                rs = reply["snippet"]
                comments.append({
                    "video_id": video_id,
                    "comment_id": reply["id"],
                    "author": rs.get("authorDisplayName"),
                    "text": rs.get("textDisplay"),
                    "like_count": rs.get("likeCount"),
                    "published_at": rs.get("publishedAt"),
                    "parent_id": item["snippet"]["topLevelComment"]["id"],
                })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return pd.DataFrame(comments)

df_youtube = pd.concat([get_youtube_comments(vid) for vid in YOUTUBE_VIDEO_IDS], ignore_index=True) if YOUTUBE_VIDEO_IDS else pd.DataFrame()
df_youtube.head()

## 5. Template scraping komentar portal berita (gratis, tapi rapuh)
Ini template umum, bukan scraper siap pakai untuk portal tertentu — struktur HTML tiap situs beda dan berubah dari waktu ke waktu. Sebelum pakai:
1. Cek `robots.txt` situs target dan gunakan secara wajar (jangan bombardir server dengan request beruntun).
2. Buka Inspect Element di browser, cari elemen yang membungkus tiap komentar, sesuaikan `selector` di bawah.
3. Kalau komentar dimuat lewat JavaScript/AJAX (banyak portal begitu), `requests` saja tidak cukup — perlu cek tab Network di browser untuk menemukan endpoint datanya, atau pakai Selenium/Playwright.

In [ ]:
from bs4 import BeautifulSoup

def scrape_comments_generic(url, comment_selector, author_selector=None, text_selector=None, delay=2):
    headers = {"User-Agent": "Mozilla/5.0 (riset akademik non-komersial)"}
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")
    results = []
    for block in soup.select(comment_selector):
        author = block.select_one(author_selector).get_text(strip=True) if author_selector and block.select_one(author_selector) else None
        text = block.select_one(text_selector).get_text(strip=True) if text_selector and block.select_one(text_selector) else block.get_text(strip=True)
        results.append({"url": url, "author": author, "text": text})
    time.sleep(delay)
    return pd.DataFrame(results)

# Contoh pemakaian (selector di bawah HARUS disesuaikan per situs, ini cuma placeholder):
# df_news = pd.concat([scrape_comments_generic(u, comment_selector=".comment-item", author_selector=".comment-author", text_selector=".comment-text") for u in NEWS_URLS], ignore_index=True)
df_news = pd.DataFrame()  # isi manual setelah selector disesuaikan

## 6. Gabungkan data & bangun graf interaksi (inti SNA)
Node = aktor (username/nama). Edge = relasi antar aktor:
- Twitter: balasan (reply) dan mention.
- YouTube: balasan ke komentar lain (pakai `parent_id`) -- jadi graf tetap bisa terbentuk walau cuma pakai data YouTube, selama ada komentar yang saling membalas (bukan cuma komentar tunggal ke video).

Catatan: komentar yang tidak dibalas siapa pun tidak akan punya edge -- itu normal, bukan bug. Video dengan banyak diskusi/perdebatan di kolom komentar akan menghasilkan graf yang lebih kaya, cocok untuk uji coba pertama yang gratis.


In [ ]:
import networkx as nx

G = nx.DiGraph()

# Edge dari balasan & mention di Twitter
if not df_twitter.empty:
    for _, row in df_twitter.iterrows():
        src = row["author"]
        if not src:
            continue
        if row.get("in_reply_to_username"):
            G.add_edge(src, row["in_reply_to_username"], relation="reply")
        for mention in (row.get("mentions") or []):
            G.add_edge(src, mention, relation="mention")

# Edge dari balasan komentar YouTube (pembalas -> penulis komentar induk)
if not df_youtube.empty:
    comment_author = dict(zip(df_youtube["comment_id"], df_youtube["author"]))
    for _, row in df_youtube.iterrows():
        if row.get("parent_id"):
            parent_author = comment_author.get(row["parent_id"])
            src = row["author"]
            if src and parent_author and src != parent_author:
                G.add_edge(src, parent_author, relation="yt_reply")

print(f"Jumlah aktor (node): {G.number_of_nodes()}")
print(f"Jumlah relasi (edge): {G.number_of_edges()}")


## 7. Metrik jaringan & deteksi komunitas

In [ ]:
import community as community_louvain  # dari paket python-louvain

degree_centrality = nx.degree_centrality(G)

top_actors = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:15]
print("Aktor paling sentral (degree centrality):")
for name, score in top_actors:
    print(f"  {name}: {score:.4f}")

G_undirected = G.to_undirected()
partition = community_louvain.best_partition(G_undirected) if G_undirected.number_of_nodes() > 0 else {}
n_communities = len(set(partition.values())) if partition else 0
print(f"\nJumlah klaster/komunitas terdeteksi: {n_communities}")

## 8. Visualisasi jaringan (interaktif)

In [ ]:
from pyvis.network import Network
from IPython.display import HTML

net = Network(height="600px", width="100%", directed=True, notebook=True, cdn_resources="in_line")
for node in G.nodes():
    cluster = partition.get(node, 0)
    net.add_node(node, label=node, group=cluster, title=f"Komunitas {cluster}")
for src, dst, data in G.edges(data=True):
    net.add_edge(src, dst, title=data.get("relation", ""))

net.write_html("jaringan_aktor.html")
# Kalau graf tidak tampil otomatis di bawah sel ini: buka folder Files di sidebar kiri Colab,
# klik kanan jaringan_aktor.html -> Download, lalu buka file itu langsung di browser.
HTML(filename="jaringan_aktor.html")


## 9. Analisis sentimen (opsional, pakai Claude API milik kamu)
Untuk hemat kuota/biaya, ambil sampel dulu (mis. 50-100 teks) sebelum menjalankan ke seluruh data.

In [ ]:
import anthropic

def classify_sentiment_batch(texts, api_key=ANTHROPIC_API_KEY, model="claude-sonnet-5"):
    if not api_key:
        print("Lewati: Anthropic API key kosong.")
        return []
    client = anthropic.Anthropic(api_key=api_key)
    results = []
    for text in texts:
        msg = client.messages.create(
            model=model,
            max_tokens=20,
            messages=[{
                "role": "user",
                "content": f"Klasifikasikan sentimen teks berikut sebagai satu kata: positif, negatif, atau netral. Jawab HANYA satu kata itu.\n\nTeks: \"{text}\""
            }]
        )
        results.append(msg.content[0].text.strip().lower())
    return results

# Contoh pemakaian pada sampel:
# sample_texts = df_twitter["text"].dropna().sample(min(50, len(df_twitter))).tolist()
# sentiments = classify_sentiment_batch(sample_texts)

## 10. Draf Hasil & Pembahasan berbantuan AI — BACA DULU

**Ini menghasilkan DRAF, bukan artikel jurnal final.** Sebelum dipakai:
- Setiap klaim empiris harus dicek ulang terhadap data asli.
- AI dilarang (lewat instruksi prompt) mengarang rujukan — bagian sitasi berisi placeholder `[SUMBER: ...]` yang wajib diisi manual dengan referensi asli.
- Cek kebijakan jurnal tujuan soal disclosure penggunaan AI dalam penulisan.
- Jangan submit draf ini apa adanya sebagai naskah — ini titik awal, bukan titik akhir.

In [ ]:
def draft_results_section(network_summary, sentiment_summary, api_key=ANTHROPIC_API_KEY, model="claude-sonnet-5"):
    if not api_key:
        print("Lewati: Anthropic API key kosong.")
        return None
    client = anthropic.Anthropic(api_key=api_key)
    system_prompt = (
        "Kamu membantu menyusun DRAF bagian Hasil dan Pembahasan artikel ilmiah berbahasa Indonesia "
        "berdasarkan ringkasan analisis jaringan sosial (SNA) dan sentimen yang diberikan. "
        "ATURAN WAJIB: (1) JANGAN mengarang kutipan, nama peneliti, judul artikel, atau tahun publikasi apa pun -- "
        "jika perlu merujuk literatur, tulis placeholder [SUMBER: topik-relevan] agar peneliti mengisi manual. "
        "(2) Jangan membuat klaim kausal dari data korelasional/deskriptif. "
        "(3) Tulis dengan jelas bahwa ini adalah draf yang memerlukan verifikasi manusia."
    )
    user_prompt = (
        f"Ringkasan jaringan:\n{network_summary}\n\n"
        f"Ringkasan sentimen:\n{sentiment_summary}\n\n"
        "Susun draf bagian Hasil dan Pembahasan (maks. 500 kata)."
    )
    msg = client.messages.create(
        model=model,
        max_tokens=1500,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return msg.content[0].text

# Contoh pemakaian:
# network_summary = f"{G.number_of_nodes()} aktor, {G.number_of_edges()} relasi, {n_communities} komunitas terdeteksi. Aktor paling sentral: {top_actors[:5]}"
# sentiment_summary = "isi ringkasan distribusi sentimen di sini"
# draft = draft_results_section(network_summary, sentiment_summary)
# print(draft)

## 11. Ekspor hasil

In [ ]:
df_twitter.to_csv("data_twitter.csv", index=False)
df_youtube.to_csv("data_youtube.csv", index=False)
df_news.to_csv("data_news.csv", index=False)
nx.write_gexf(G, "jaringan_aktor.gexf")  # bisa dibuka di Gephi untuk visualisasi lanjutan
print("Selesai. File CSV, GEXF, dan HTML jaringan tersimpan di sesi Colab ini -- unduh manual (folder Files di sidebar kiri) sebelum sesi berakhir.")

## Langkah selanjutnya
- Kalau alur ini sudah stabil untuk beberapa studi kasus kelas, baru pertimbangkan pindah ke VPS dengan cron job untuk pemantauan berkelanjutan (bukan sekadar snapshot per sesi).
- Untuk Facebook/Instagram: ajukan akses Meta Content Library lewat institusi (proses aplikasi, bukan instan) — jangan scraping langsung.
- Keterbatasan akses API pasca-2023 ini sendiri bisa jadi bahan diskusi metodologis yang relevan di kelas, bukan cuma hambatan teknis.